# Video Generation — PCB Conveyor Simulation

Generates a simulated conveyor feed from the **raw** dataset, for use as the video
input to the preprocessing pipeline.

Run this **before** `ImagePreprocessing.ipynb`, which consumes the video it produces.

The source must be the raw dataset, not the preprocessed one. Feeding already-processed
images here would run the pipeline twice; measured on a real board, a second pass
dropped edge sharpness by 89%.

## 1. Imports and configuration

In [ ]:
import os
import cv2
import random
import shutil
import subprocess
import numpy as np

# ---------- paths ----------
INPUT_FOLDER = "../Clean_Dataset"        # RAW images
VIDEO_NAME   = "PCB_Conveyor.mp4"        # ImagePreprocessing.ipynb reads this

# ---------- board size ----------
# None = use the dataset's native resolution. ImagePreprocessing.ipynb derives the
# same value the same way, so the two notebooks cannot drift apart.
BOARD_SIZE_OVERRIDE = None

# ---------- frame ----------
WIDTH, HEIGHT = 1280, 720
FPS           = 60
DURATION_SEC  = 20

# ---------- motion ----------
SPEED       = 4.0      # pixels per frame -> SPEED * FPS px/second
BELT_MARGIN = 40       # belt padding above and below the board

# ---------- encoding ----------
CRF       = 18         # H.264 quality: lower is better, 18 is near-lossless
POOL_SIZE = 60         # how many different boards to cycle through

print("Input :", INPUT_FOLDER)
print("Output:", VIDEO_NAME)

Input : ../Clean_Dataset
Output: PCB_Conveyor3.mp4


## 2. Collect the image paths

In [2]:
classes = sorted(
    folder for folder in os.listdir(INPUT_FOLDER)
    if os.path.isdir(os.path.join(INPUT_FOLDER, folder))
)

image_paths = []

for cls in classes:
    folder = os.path.join(INPUT_FOLDER, cls)

    for file in sorted(os.listdir(folder)):
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            image_paths.append(os.path.join(folder, file))

print("Classes:", classes)
print("Total images:", len(image_paths))

Classes: ['Missing_hole_rotation', 'Mouse_bite_rotation', 'Open_circuit_rotation', 'Short_rotation', 'Spur_rotation', 'Spurious_copper_rotation']
Total images: 693


## 3. Board size and belt geometry

The belt geometry is derived from `BOARD_SIZE` rather than hard-coded, so changing the
board size cannot silently push the board off the belt or off the frame.

In [3]:
def resolve_board_size(folder, override=None):
    """
    Board resolution, derived from the dataset itself.

    Both notebooks call this, so neither has to be told what the other used.
    Rendering boards smaller than native and upscaling them later cannot recover
    the lost detail: measured on a 512 px dataset, rendering at 180 px and
    upscaling back retained only 6.5% of the original edge sharpness.
    """
    if override is not None:
        print(f"BOARD_SIZE set explicitly: {override} px")
        return override

    for cls in sorted(os.listdir(folder)):
        cls_path = os.path.join(folder, cls)

        if not os.path.isdir(cls_path):
            continue

        for file in sorted(os.listdir(cls_path)):
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                probe = cv2.imread(os.path.join(cls_path, file))

                if probe is not None:
                    size = min(probe.shape[:2])
                    print(f"BOARD_SIZE taken from the dataset: {size} px")
                    return size

    raise RuntimeError(f"No readable images found in {folder}")


BOARD_SIZE = resolve_board_size(INPUT_FOLDER, BOARD_SIZE_OVERRIDE)

belt_height = BOARD_SIZE + 2 * BELT_MARGIN

if belt_height > HEIGHT:
    raise ValueError(
        f"BOARD_SIZE={BOARD_SIZE} plus margins needs {belt_height}px but the frame is "
        f"only {HEIGHT}px tall. Raise HEIGHT to at least {belt_height}, or set "
        f"BOARD_SIZE_OVERRIDE to something smaller."
    )

BELT_TOP    = (HEIGHT - belt_height) // 2
BELT_BOTTOM = BELT_TOP + belt_height
BELT_Y      = (BELT_TOP + BELT_BOTTOM) // 2 - BOARD_SIZE // 2

SPACING    = int(BOARD_SIZE * 1.25)      # must exceed BOARD_SIZE or boards overlap
STRIPE_GAP = max(8, SPACING // 4)

n_slots = int(np.ceil((WIDTH + 2 * SPACING) / SPACING))
track   = n_slots * SPACING              # wrapping by this keeps spacing exact

# Snap to a whole number of belt cycles so the video loops seamlessly
cycle_frames = track / SPEED
TOTAL_FRAMES = int(round(max(1, round(FPS * DURATION_SEC / cycle_frames)) * cycle_frames))

print(f"Board         : {BOARD_SIZE}x{BOARD_SIZE} px at y {BELT_Y}..{BELT_Y + BOARD_SIZE}")
print(f"Belt          : y {BELT_TOP}..{BELT_BOTTOM}  (frame height {HEIGHT})")
print(f"Spacing       : {SPACING} px   ->  ~{WIDTH // SPACING} boards visible")
print(f"Slots on belt : {n_slots}")
print(f"Total frames  : {TOTAL_FRAMES} ({TOTAL_FRAMES / FPS:.1f} s)")

BOARD_SIZE taken from the dataset: 512 px
Board         : 512x512 px at y 104..616
Belt          : y 64..656  (frame height 720)
Spacing       : 640 px   ->  ~2 boards visible
Slots on belt : 4
Total frames  : 1280 (21.3 s)


## 4. Load the boards once

Loading and resizing before the render loop keeps the loop cheap. `INTER_AREA` is
correct for downscaling and `INTER_CUBIC` for enlarging; at native resolution neither
runs at all.

In [4]:
random.seed(42)
random.shuffle(image_paths)

pool = []
resized_count = 0

for path in image_paths:
    img = cv2.imread(path)

    if img is None:
        continue

    if img.shape[0] != BOARD_SIZE or img.shape[1] != BOARD_SIZE:
        interp = cv2.INTER_AREA if img.shape[0] > BOARD_SIZE else cv2.INTER_CUBIC
        img = cv2.resize(img, (BOARD_SIZE, BOARD_SIZE), interpolation=interp)
        resized_count += 1

    pool.append(img)

    if len(pool) >= POOL_SIZE:
        break

print(f"Loaded {len(pool)} PCB images at {BOARD_SIZE}x{BOARD_SIZE}")
print("  (all at native resolution, no detail lost)" if not resized_count
      else f"  ({resized_count} were resized)")

Loaded 60 PCB images at 512x512
  (all at native resolution, no detail lost)


## 6. Static background + moving belt texture

The background never changes, so build it once and `.copy()` it each frame.
The diagonal stripes *do* move — they give the eye a motion reference, which is a
large part of why a conveyor reads as smooth rather than as boards sliding on nothing.

In [5]:
background = np.full((HEIGHT, WIDTH, 3), 255, np.uint8)   # 255 = white, lower = grey

# belt surface
cv2.rectangle(background, (0, BELT_TOP), (WIDTH, BELT_BOTTOM), (95, 95, 95), -1)

# side rails
cv2.rectangle(background, (0, BELT_TOP), (WIDTH, BELT_TOP + 14), (58, 58, 58), -1)
cv2.rectangle(background, (0, BELT_BOTTOM - 14), (WIDTH, BELT_BOTTOM), (58, 58, 58), -1)

# thin outline so the belt reads cleanly against a white background
cv2.rectangle(background, (0, BELT_TOP), (WIDTH - 1, BELT_BOTTOM), (150, 150, 150), 2)


def draw_belt_texture(frame, offset):
    x = -STRIPE_GAP + (offset % STRIPE_GAP)

    while x < WIDTH + STRIPE_GAP:
        cv2.line(
            frame,
            (int(x), BELT_TOP + 14),
            (int(x) + 26, BELT_BOTTOM - 14),
            (78, 78, 78), 3, cv2.LINE_AA
        )
        x += STRIPE_GAP

## 7. Paste helper

`cv2.rectangle` clips itself at the frame border, but a raw
NumPy slice assignment does not — that is what the manual clipping below is for.

In [6]:
def paste(frame, img, x, y):
    h, w = img.shape[:2]

    x0 = max(0, x)
    x1 = min(WIDTH, x + w)

    if x1 <= x0:
        return

    frame[y:y + h, x0:x1] = img[:, x0 - x:x1 - x]

## 8. Open the video writer

`cv2.VideoWriter` gives you **no control over bitrate** — `set(VIDEOWRITER_PROP_QUALITY, ...)`
returns `False` and does nothing. Whatever your build's encoder defaults to is what you get,
and some builds default to near-lossless (hundreds of MB for a 20 s clip), which is exactly
what makes playback stutter.

So we pipe raw frames into `ffmpeg` instead and set the quality with `-crf`. Same fidelity,
roughly 30x smaller. If `ffmpeg` is not on PATH we fall back to `cv2.VideoWriter`.

In [7]:
import shutil
import subprocess


class FFmpegWriter:
    """Pipes raw BGR frames into ffmpeg so we can control quality with -crf."""

    def __init__(self, path, fps, size, crf=18):
        w, h = size
        self.proc = subprocess.Popen(
            [
                "ffmpeg", "-y", "-loglevel", "error",
                "-f", "rawvideo", "-pix_fmt", "bgr24",
                "-s", f"{w}x{h}", "-r", str(fps), "-i", "-",
                "-c:v", "libx264", "-preset", "medium", "-crf", str(crf),
                "-pix_fmt", "yuv420p",      # required or some players reject the file
                "-movflags", "+faststart",
                path,
            ],
            stdin=subprocess.PIPE,
        )

    def write(self, frame):
        self.proc.stdin.write(frame.tobytes())

    def release(self):
        self.proc.stdin.close()
        self.proc.wait()


def open_writer(name, fps, size, crf=CRF):
    if shutil.which("ffmpeg"):
        print(f"Using ffmpeg (libx264, crf={crf})")
        return FFmpegWriter(name, fps, size, crf)

    print("ffmpeg not found on PATH - falling back to cv2.VideoWriter.")
    print("The file will be MUCH larger and may stutter. Install ffmpeg to fix.")

    log = cv2.utils.logging
    prev = log.getLogLevel()
    log.setLogLevel(log.LOG_LEVEL_SILENT)

    try:
        for tag in ("avc1", "H264", "mp4v"):
            writer = cv2.VideoWriter(name, cv2.VideoWriter_fourcc(*tag), fps, size)

            if writer.isOpened():
                print("  codec:", tag)
                return writer

            writer.release()
    finally:
        log.setLogLevel(prev)

    raise RuntimeError("No usable codec found")


video = open_writer(VIDEO_NAME, FPS, (WIDTH, HEIGHT))

ffmpeg not found on PATH - falling back to cv2.VideoWriter.
The file will be MUCH larger and may stutter. Install ffmpeg to fix.
  codec: avc1


## 9. Render

In [8]:
slots = [
    {"x": float(i * SPACING - SPACING), "idx": i % len(pool)}
    for i in range(n_slots)
]

next_idx = n_slots % len(pool)

for frame_id in range(TOTAL_FRAMES):

    frame = background.copy()
    draw_belt_texture(frame, frame_id * SPEED)

    for slot in slots:

        slot["x"] += SPEED

        # wrap by exactly one track length -> spacing is preserved
        if slot["x"] > WIDTH:
            slot["x"] -= track
            slot["idx"] = next_idx
            next_idx = (next_idx + 1) % len(pool)

        x = int(round(slot["x"]))
        img = pool[slot["idx"]]

        # drop shadow
        cv2.rectangle(
            frame,
            (x + 7, BELT_Y + 9),
            (x + BOARD_SIZE + 7, BELT_Y + BOARD_SIZE + 9),
            (66, 66, 66), -1
        )

        paste(frame, img, x, BELT_Y)

        cv2.rectangle(
            frame,
            (x - 1, BELT_Y - 1),
            (x + BOARD_SIZE, BELT_Y + BOARD_SIZE),
            (30, 30, 30), 2
        )

    video.write(frame)

    if (frame_id + 1) % 200 == 0:
        print(f"  {frame_id + 1}/{TOTAL_FRAMES} frames")

  200/1280 frames
  400/1280 frames
  600/1280 frames
  800/1280 frames
  1000/1280 frames
  1200/1280 frames


### Save the video

In [9]:
video.release()

size_mb = os.path.getsize(VIDEO_NAME) / 1e6

print("Video saved:", VIDEO_NAME)
print(f"Size: {size_mb:.1f} MB")

if size_mb > 60:
    print("\nThat file is far larger than it should be (expect roughly 5-20 MB).")
    print("ffmpeg was not found, so OpenCV's encoder ran at near-lossless quality.")
    print("Install ffmpeg and re-run - the size itself is what makes playback stutter.")

Video saved: PCB_Conveyor3.mp4
Size: 294.9 MB

That file is far larger than it should be (expect roughly 5-20 MB).
ffmpeg was not found, so OpenCV's encoder ran at near-lossless quality.
Install ffmpeg and re-run - the size itself is what makes playback stutter.


## Notes

**Expected file size** is roughly 5-20 MB for a 20 s clip. If the save step warned that
ffmpeg was missing and the file is hundreds of MB, install ffmpeg and re-run — that
size is what makes playback stutter, not the frame rate.

- Windows: `winget install ffmpeg`
- macOS: `brew install ffmpeg`
- Linux: `sudo apt install ffmpeg`

To re-encode an oversized file instead of re-rendering:

```
ffmpeg -i PCB_Conveyor.mp4 -c:v libx264 -crf 18 -pix_fmt yuv420p PCB_Conveyor_small.mp4
```

**Fewer boards visible at larger board sizes** is expected: a 512 px board on a 1280 px
frame leaves room for about two. Raise `WIDTH`/`HEIGHT` if more are needed on screen.